# ACAPS Risk Radar — Data Exploration

This notebook walks you through the ACAPS Risk Radar dataset provided for **Hackathon for Good #8**.

By the end, you will:
- Understand the structure and content of the dataset
- See the distribution of risks by country, level, and status
- Explore the demo risk (R1043 — Chad/Sudan)
- Validate the materialisation rates mentioned in the challenge brief
- Have a foundation to build your solution on

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("../data/risk-radar-export.csv")
print(f"Dataset: {len(df)} risks, {df['Country'].nunique()} countries, {len(df.columns)} columns")
df.head(3)

## 2. Dataset Overview

In [ ]:
# Column overview: name, non-null count, and sample values
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = non_null / len(df) * 100
    print(f"{col:45s}  {non_null:3d}/{len(df)} ({pct:.0f}%)")

## 3. Risk Level Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Risk Level
colors_level = {"High": "#d32f2f", "Medium": "#f57c00", "Low": "#388e3c"}
level_counts = df["Risk Level"].value_counts()
level_counts.plot.bar(ax=axes[0], color=[colors_level.get(x, "#999") for x in level_counts.index])
axes[0].set_title("Risk Level Distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=0)

# Status
colors_status = {"Active": "#1976d2", "Materialised": "#d32f2f", "Not materialised": "#388e3c"}
status_counts = df["Status"].value_counts()
status_counts.plot.bar(ax=axes[1], color=[colors_status.get(x, "#999") for x in status_counts.index])
axes[1].set_title("Status Distribution")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=15)

# Risk Type
type_counts = df["Risk Type"].value_counts()
type_counts.plot.barh(ax=axes[2], color="#5c6bc0")
axes[2].set_title("Risk Type Distribution")
axes[2].set_xlabel("Count")

plt.tight_layout()
plt.show()

## 4. Geographic Distribution

In [ ]:
# Top 15 countries by number of risks
top_countries = df["Country"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
top_countries.plot.barh(ax=ax, color="#5c6bc0")
ax.set_title("Top 15 Countries by Number of Risks")
ax.set_xlabel("Number of Risks")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Materialisation Analysis

The challenge brief states:
> *"Roughly eight in ten of those ACAPS rated 'High' materialised, against about four in ten of those rated 'Low'."*

Let's validate this against the data.

In [ ]:
# Filter to resolved risks (materialised or not materialised)
resolved = df[df["Status"].isin(["Materialised", "Not materialised"])].copy()
resolved["Did materialise"] = resolved["Status"] == "Materialised"

print(f"Resolved risks: {len(resolved)} out of {len(df)} total")
print()

# Materialisation rate by risk level
mat_rate = resolved.groupby("Risk Level")["Did materialise"].agg(["sum", "count"])
mat_rate["rate"] = mat_rate["sum"] / mat_rate["count"]
mat_rate.columns = ["Materialised", "Total", "Rate"]

for level in ["High", "Medium", "Low"]:
    if level in mat_rate.index:
        row = mat_rate.loc[level]
        print(f"{level:8s}: {row['Materialised']:.0f}/{row['Total']:.0f} materialised ({row['Rate']:.0%})")

print()
print("These rates can be used to calibrate and test a scoring method for your 'super estimator' scoreboard.")

## 6. The Demo Risk: R1043 (Chad / Sudan)

The challenge brief suggests building your demo around this risk:
> *"Escalating cross-border insecurity linked to the Sudan conflict increases protection risks and humanitarian needs in eastern Chad"*

In [ ]:
r1043 = df[df["Risk ID"] == "R1043"].iloc[0]

print(f"Risk ID:          {r1043['Risk ID']}")
print(f"Title:            {r1043['Risk Title']}")
print(f"Country:          {r1043['Country']}")
print(f"Risk Type:        {r1043['Risk Type']}")
print(f"Risk Level:       {r1043['Risk Level']}")
print(f"Probability:      {r1043['Probability']}")
print(f"Impact:           {r1043['Impact']}")
print(f"Status:           {r1043['Status']}")
print(f"Risk Trend:       {r1043['Risk trend']}")
print(f"Expected Exposure:{r1043['Expected exposure']}")
print(f"Priority Needs:   {r1043['Expected priority needs']}")
print()
print("--- Rationale (first 500 chars) ---")
print(str(r1043['Rationale'])[:500])

In [ ]:
# What triggers are identified for R1043?
triggers_raw = str(r1043["Triggers"])
print("--- Triggers ---")
print(triggers_raw[:1000])

## 7. Parsing Triggers

The triggers field has a structured format that you might want to parse for your solution.

In [ ]:
import re

def parse_triggers(text):
    """Parse the structured triggers field into a list of dicts."""
    if pd.isna(text):
        return []
    
    triggers = []
    # Split on 'Trigger ID:' boundaries
    parts = re.split(r'(?=Trigger ID:)', text)
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
        
        trigger = {}
        
        id_match = re.search(r'Trigger ID:\s*(.+?)\n', part)
        if id_match:
            trigger['id'] = id_match.group(1).strip()
        
        title_match = re.search(r'Title:\s*(.+?)\n', part)
        if title_match:
            trigger['title'] = title_match.group(1).strip()
        
        desc_match = re.search(r'Description:\s*\n?(.+)', part, re.DOTALL)
        if desc_match:
            trigger['description'] = desc_match.group(1).strip()[:200]
        
        if trigger:
            triggers.append(trigger)
    
    return triggers

# Example: parse R1043's triggers
parsed = parse_triggers(r1043["Triggers"])
for t in parsed:
    print(f"  {t.get('id', '?')}: {t.get('title', '?')}")

## 8. Expected Exposure Distribution

In [ ]:
exposure_order = ["<20,000", "20,000-100,000", "100,000-250,000", "250,000-500,000", ">500,000"]
exposure_counts = df["Expected exposure"].value_counts().reindex(exposure_order).fillna(0)

fig, ax = plt.subplots(figsize=(8, 4))
exposure_counts.plot.bar(ax=ax, color="#e57373")
ax.set_title("Expected Exposure Distribution")
ax.set_ylabel("Number of Risks")
ax.set_xlabel("People Potentially Affected")
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 9. Priority Needs Analysis

In [ ]:
# Priority needs are comma-separated — explode and count
needs = df["Expected priority needs"].dropna().str.split(",").explode().str.strip()
needs_counts = needs.value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
needs_counts.plot.barh(ax=ax, color="#4db6ac")
ax.set_title("Most Common Priority Needs Across All Risks")
ax.set_xlabel("Number of Risks")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Probability vs Impact

Visualising the risk matrix from the ACAPS methodology.

In [ ]:
# Map probability to numeric
prob_map = {"Very Low": 1, "Low": 2, "Medium": 3, "High": 4, "Very High": 5}
plot_df = df.dropna(subset=["Probability", "Impact"]).copy()
plot_df["Probability_num"] = plot_df["Probability"].map(prob_map)
plot_df["Impact_num"] = pd.to_numeric(plot_df["Impact"], errors="coerce")
plot_df = plot_df.dropna(subset=["Probability_num", "Impact_num"]).reset_index(drop=True)

color_map = {"High": "#d32f2f", "Medium": "#f57c00", "Low": "#388e3c"}
colors = plot_df["Risk Level"].map(color_map).fillna("#999")

# Add jitter to avoid overlapping points
jitter = (plot_df.index % 5 - 2) * 0.05

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    plot_df["Impact_num"] + jitter,
    plot_df["Probability_num"] + jitter,
    c=colors,
    alpha=0.6,
    s=60,
    edgecolors="white",
    linewidth=0.5,
)
ax.set_xlabel("Impact (1-5)")
ax.set_ylabel("Probability")
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["Very Low", "Low", "Medium", "High", "Very High"])
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_title("Risk Matrix: Probability vs Impact")
ax.set_xlim(0.5, 5.5)
ax.set_ylim(0.5, 5.5)

# Legend
for level, color in color_map.items():
    ax.scatter([], [], c=color, s=60, label=level)
ax.legend(title="Risk Level", loc="upper left")

plt.tight_layout()
plt.show()

## Next Steps

Now that you understand the data, here are some directions to explore:

1. **Ground Truthing:** How would you present a risk like R1043 to an expert for validation? What specific questions would you ask?

2. **Forecasting:** Can you turn the triggers into yes/no forecast questions with a six-month horizon? How would you score experts' predictions against actual outcomes?

3. **Scoring:** Use the materialisation rates to simulate expert predictions and build a Brier score or similar calibration metric.

4. **Anonymity:** Design the data model for expert contributions with tiered anonymity (known to ACAPS, pseudonymous to peers, anonymous to public).

Good luck! Remember: **Alexander Harper from ACAPS is on site** — use that access.